In [1]:
# 토지 소설을 글자 단위로 학습한 후 소설 쓰기

import tensorflow as tf
import numpy as np
import re
import urllib.request
import random

# 1) 데이터 로드
url = 'https://raw.githubusercontent.com/pykwon/etc/refs/heads/master/rnn_short_toji.txt'
with urllib.request.urlopen(url) as response:
    text = response.read().decode('utf-8')

print(text[:100])
print('텍스트 총 몇 자?', len(text))

귀녀의 모습을 한번 쳐다보고 떠나려 했다. 집안을 이리저리 기웃거리던 강표수는 윤씨부
인에게 인사를 올리고 중문을 나서는  치수 뒷모습을 보았다. 실망에  얼굴이 일그러지면서 
텍스트 총 몇 자? 351744


In [2]:
# 2) 텍스트 정제 (한글+공백만 남김, 필요시 규칙 조정)
# text = re.sub(r"[^가-힣 ]", " ", text)
text = re.sub(r"[^가-힣 .,?!]", " ", text)  # 메모리/품질 조절용 대안
text = re.sub(r"\s{2,}", " ", text).strip()

# 고유 문자/매핑
chars = sorted(list(set(text)))
print('사용 가능한 문자 수 : ', len(chars))
char2idx = {c:i for i,c in enumerate(chars)}
idx2char = np.array(chars, dtype=object)

print(list(char2idx.items())[:5])
print(list(enumerate(chars))[:5])

사용 가능한 문자 수 :  1427
[(' ', 0), ('!', 1), (',', 2), ('.', 3), ('?', 4)]
[(0, ' '), (1, '!'), (2, ','), (3, '.'), (4, '?')]


In [3]:
# 3) 학습 시퀀스 생성 (슬라이딩 윈도우)
maxlen = 30   # 입력 길이
step = 10     # 샘플 간 간격(큼=적은 샘플=메모리↓)
sentences = []
next_chars = []

for i in range(0, len(text) - maxlen, pos_step := step):
    sentences.append(text[i:i+maxlen])
    next_chars.append(text[i+maxlen])

print('시퀀스 갯수:', len(sentences))

시퀀스 갯수: 33012


In [4]:
# 4) 벡터화 (정수 인덱스만! one-hot 생성하지 않음)
N = len(sentences)
vocab_size = len(chars)

# 입력: (N, maxlen) int32
x_idx = np.zeros((N, maxlen), dtype=np.int32)
for i, s in enumerate(sentences):
    for t, ch in enumerate(s):
        x_idx[i, t] = char2idx[ch]

# 라벨: (N,) int32 – 다음 글자의 인덱스
y_idx = np.array([char2idx[c] for c in next_chars], dtype=np.int32)

print('x_idx.shape =', x_idx.shape, ' y_idx.shape =', y_idx.shape)

x_idx.shape = (33012, 30)  y_idx.shape = (33012,)


In [5]:
# 5) 모델 구성: Embedding + LSTM + Dense
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout

model = Sequential([
    Input(shape=(maxlen,)),                              # (batch, time)
    Embedding(input_dim=vocab_size, output_dim=128),     # char index -> vector (메모리 매우 절감)
    LSTM(256, return_sequences=True),
    Dropout(0.2),
    LSTM(128),
    Dropout(0.2),
    Dense(vocab_size, activation='softmax')  # 클래스 개수 = vocab_size
])

# SparseCategoricalCrossentropy: y가 one-hot이 아니라 '정수 인덱스'일 때 사용
opt = tf.keras.optimizers.RMSprop(learning_rate=0.003)   # 0.01 → 0.003 정도가 안정적
model.compile(optimizer=opt, loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['sparse_categorical_accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 30, 128)        │       182,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 30, 256)        │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1427)           │       184,083 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 958,099 (3.65 MB)

 Trainable params: 958,099 (3.65 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# 6) 학습 (정수 텐서만 배치로 메모리에 올리므로 매우 가볍습니다)
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
ckpt  = ModelCheckpoint('bestmodel_sparse.keras', monitor='val_loss', save_best_only=True)
early = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit( x_idx, y_idx,
    batch_size=256, epochs=500, validation_split=0.05, callbacks=[ckpt, early],
    verbose=2
)

# 모델 읽기 (추론만이면 compile=False가 더 빠르고 안전)
from tensorflow.keras.models import load_model
model = load_model('bestmodel_sparse.keras', compile=False)

start_index = random.randint(0, len(text) - maxlen - 1)
seed_text = text[start_index : start_index + maxlen]

# 샘플링 & 텍스트 생성 유틸 ---------
def sample_from_probs(probs, temperature=0.7):
    probs = np.asarray(probs, dtype=np.float64)
    probs = np.maximum(probs, 1e-12)
    logits = np.log(probs) / max(temperature, 1e-8)
    logits -= np.max(logits)
    p = np.exp(logits)
    p /= p.sum()
    return np.random.choice(len(p), p=p)

def make_context_indices(seed_text, maxlen, char2idx, pad_char=' '):
    """seed 길이가 maxlen보다 짧으면 왼쪽을 pad_char로 채워 길이를 맞춤"""
    if len(seed_text) < maxlen:
        seed_text = (pad_char * (maxlen - len(seed_text))) + seed_text
    else:
        seed_text = seed_text[-maxlen:]
    # 사전에 없는 문자는 pad_char로 대체
    return np.array([char2idx.get(ch, char2idx.get(pad_char, 0)) for ch in seed_text], dtype=np.int32)

def generate_text(model, seed_text, length=400, temperature=0.7):
    """
    model : 학습/로드된 Keras 모델
    seed_text : 시작 시드 문자열 (길이가 maxlen보다 짧아도 됨)
    length : 생성할 글자 수
    temperature : 샘플링 온도 (0.5~1.2 범위에서 실험 추천)
    """
    context_idx = make_context_indices(seed_text, maxlen, char2idx, pad_char=' ')
    generated = [ch for ch in seed_text]  # 결과 누적

    for _ in range(length):
        x = context_idx[np.newaxis, :]          # (1, maxlen)
        # model 출력: (1, vocab_size)  — 마지막 LSTM의 softmax 결과
        probs = model.predict(x, verbose=0)[0]             # 확률 벡터
        next_id = sample_from_probs(probs, temperature=temperature)
        next_ch = idx2char[next_id]
        generated.append(next_ch)

        # 슬라이딩 윈도우 갱신
        context_idx = np.concatenate([context_idx[1:], [next_id]])

    return ''.join(generated)

Epoch 1/500
123/123 - 8s - 64ms/step - loss: 4.7954 - sparse_categorical_accuracy: 0.2487 - val_loss: 4.3277 - val_sparse_categorical_accuracy: 0.2586
Epoch 2/500
123/123 - 2s - 16ms/step - loss: 4.3134 - sparse_categorical_accuracy: 0.2582 - val_loss: 4.0922 - val_sparse_categorical_accuracy: 0.2653
Epoch 3/500
123/123 - 2s - 16ms/step - loss: 4.1762 - sparse_categorical_accuracy: 0.2607 - val_loss: 4.0441 - val_sparse_categorical_accuracy: 0.2714
Epoch 4/500
123/123 - 2s - 18ms/step - loss: 4.1145 - sparse_categorical_accuracy: 0.2685 - val_loss: 4.0103 - val_sparse_categorical_accuracy: 0.2823
Epoch 5/500
123/123 - 2s - 17ms/step - loss: 4.0419 - sparse_categorical_accuracy: 0.2805 - val_loss: 3.9238 - val_sparse_categorical_accuracy: 0.2907
Epoch 6/500
123/123 - 2s - 16ms/step - loss: 3.9832 - sparse_categorical_accuracy: 0.2857 - val_loss: 3.8725 - val_sparse_categorical_accuracy: 0.2986
Epoch 7/500
123/123 - 2s - 16ms/step - loss: 3.9188 - sparse_categorical_accuracy: 0.2910 - va

In [7]:
# 실제 생성 실행 ---------
# seed_text 준비
print("\n=== 샘플 생성 ===")
print("Seed:", seed_text.replace("\n", " "))
gen = generate_text(model, seed_text, length=400, temperature=0.8)
print(gen)   # 훈련량/샘플 밀도 부족으로 결과는 불만

save_path = "generated_toji.txt"    # 저장할 파일 이름
with open(save_path, "w", encoding="utf-8") as f:
    f.write(seed_text + "\n\n")
    f.write("=== 생성 결과 ===\n")
    f.write(gen)

print(f"[완료] 결과가 '{save_path}' 파일로 저장")


=== 샘플 생성 ===
Seed: 다. 흐음...... 언제까지 이러고 있을 것인가. 우
다. 흐음...... 언제까지 이러고 있을 것인가. 우디 나니께. 아도 사도네 눈으로 한 일이었다. 그만이 우리 그 일 좋이 날 마가. 하나라. 발을 얘아보고 좋었다. 마리 와밀은 우관이 이 묵는 말을 내겠다. 주상이 좋어서 있어서 이 부린 다리가 여도 있을 인 있고 마음이 없었다. 막명 빌이 왔다.다. 마음과 꾸망선을 다음정이 있었다. 기음이 모궅소 않문이었다. 사사이이 지리 있어갔다? 어서 있는 예런. 어미 그러지 다리! 허로 그.가 그러나. 월씨 용이는 어신 죽은 내는 이고 란식가 나니다. 사안이는 목둥이어놓았다. 그럴 무당 지리미 것이지마. 신자의 조준막를 지타질 일 마는 것도 눈이다 그 동랑 을 나는 자을 내기 아오. 말 아니니, 아니어! 한 때 말이가 한 일네. 아이앉은 강포수가 죽은 사안의 말을 넘어볼을, 조튼 봉순의 윤청댁은 행적이라 강포수는 날
[완료] 결과가 'generated_toji.txt' 파일로 저장
